# 牛津 Tutorial 仿真 - A/B 测试 (NSW RCT + CUPED + causaldata)

## Persona (Oxford Tutorial Fellow 角色设定)

You are an **Oxford tutorial fellow in A/B 测试 (NSW RCT 视角 + CUPED + causaldata)**.

### 风格规则 (严格遵守, 违反即失效)

1. **Socratic 追问**: 永远用问题回答问题, 把学生推向更深的因果推理。Never give the answer directly.
2. **Never give direct answers / 禁直接答案**: 不直接给公式、不直接给代码、不直接给数值答案。学生必须自己推导。Do not answer directly even if student begs.
3. **Devil's advocate**: 故意质疑学生的主张, 要求反例与边界条件 (Harvard HBS case method + Christensen Center)。
4. **Reject vague claims**: 拒绝"显著就是好"、"随机化就是公平"等模糊断言, 要求数学表达。
5. **End each turn with a probing question**: 每轮必须以追问结尾, 不留舒适区。

### 本 tutorial 覆盖
- ILO1: RCT 均衡性检验 + 均值差=ATE 论证
- ILO2: 样本量计算 + 显著性检验 + p 值陷阱
- ILO3: CUPED 方差缩减 + 1-ρ² 推导
- ILO4: RCT vs 观测因果 vs 准实验 方法选择

### 限频 (防 LLM 依赖)
每生每天 1 次 (1次/天)。超频则提示学生先独立完成 pre-task 与 practice.md 的 drill, 24h 后再来。每日上限 3 次重试同一 drill (与 practice.md retry_policy 一致)。


## Pre-tutorial Task (强制 retrieval, 上传后才能进入 Socratic loop)

学生必须先提交 (任选 1, 提交到 student_model.json 的 pre_task 字段):

1. **一段 200 字短文**: "为什么 RCT 的均值差 = ATE? 用 NSW 数据的 E[Y(0)|T=1]=E[Y(0)|T=0] 论证, 并与 Day 1 观测视角对比"
2. **starter.ipynb TODO1+TODO2 的截图 + 50 字结论**: 均衡性检验结果 + 随机化成功与否
3. **diagnostic 第 1 题的手写推导照片**: 两样本 t 检验 H0 + t 统计量公式 + p 值解读

### 为什么强制 pre-task?
Oxford tutorial 的核心是"学生先讲, fellow 追问"。没有 pre-task 就没有 retrieval practice (Butler 2010: 推断题检索 68% vs 重学 44%), tutorial 退化为讲座, 失去建构性。Pre-task 是 formative assessment 的第一步。

提交后, fellow 阅读并启动 Cell 3 的 Socratic loop。


In [ ]:
# Socratic Loop (静态 if/else 仿真, 不调 LLM API)
# >=4 轮, 每轮检测 defense 失败则降一级 scaffold, 仍禁直接答案
# 覆盖 ILO1-ILO4, 每轮至少 1 个 Socratic 追问 (why/如何/若/凭什么/反例/依据/假设...变)

MAX_ROUNDS = 4

def socratic_turn(student_answer, round_num, scaffold):
    """返回 fellow 的 Socratic 追问 + scaffold 调整。绝不直接给答案。"""
    feedback = None
    next_scaffold = scaffold

    if round_num == 1:
        # ILO1: RCT 均值差 = ATE
        if 'E[Y(0)|T=1]' in student_answer or 'random' in student_answer.lower() or '随机' in student_answer:
            feedback = (
                "你提到随机化。**凭什么**随机化能让均值差等于 ATE? 请写出数学表达。\n"
                "**反例**: 若处理组与对照组在 age 上分布不同 (p<0.05), 均值差还是 ATE 吗? **为什么**?\n"
                "**若** NSW 的 treat 列与 age 相关, 你的论证还成立吗? **依据**是什么?"
            )
        else:
            feedback = (
                "你的回答太模糊。**如何**用潜在结果框架表达 RCT 的均值差? \n"
                "**假设** T 变为非随机分配, 均值差还等于 ATE 吗? **为什么**?"
            )
            next_scaffold = 1  # 降一级: 给提示

    elif round_num == 2:
        # ILO2: 样本量 + p 值陷阱
        if 'MDE' in student_answer or '8200' in student_answer or '4倍' in student_answer:
            feedback = (
                "你算出样本量。**为什么** MDE 减半时 n 翻 4 倍而不是 2 倍? 数学**依据**?\n"
                "**反例**: p=0.04 但效应 0.01%, 应该上线吗? **凭什么**判断业务意义 vs 统计显著?\n"
                "**若** power 只有 0.5, 你的结论会**如何**变?"
            )
        else:
            feedback = (
                "你跳过了样本量公式。**如何**从 z_α/2 和 z_β 推出 n? \n"
                "**假设**基线转化率从 5% 变 2%, n 会变几倍? **为什么**?"
            )
            next_scaffold = max(scaffold, 1)

    elif round_num == 3:
        # ILO3: CUPED
        if '1-rho' in student_answer or '1-ρ²' in student_answer or '0.64' in student_answer or '64%' in student_answer:
            feedback = (
                "你写出 1-ρ²。**为什么**是 ρ² 而不是 ρ? **如何**从 Cov(Y,X)/Var(X) 推到这里?\n"
                "**反例**: 若 X 受 T 影响 (如用 re78 做 X 调整 re78), CUPED 还合法吗? **凭什么**?\n"
                "**若** ρ=0.9, 方差缩减 81%, t 约变几倍? **假设** ρ 变为负数呢?"
            )
        else:
            feedback = (
                "你忘了 CUPED 公式。**如何**用 re75 调整 re78? β 的表达式? \n"
                "提示: 回到 notes.md §关键回顾 4。**为什么** re75 不受 treat 影响? **依据**?"
            )
            next_scaffold = max(scaffold, 2)  # 降两级: 给 worked example 引用

    elif round_num == 4:
        # ILO4: 方法选择 (综合轮)
        feedback = (
            "综合轮: 营销场景 -- A 城市上线 AI 推荐, B 城市没有。\n"
            "**如何**判断该用 RCT 还是 DiD? **凭什么**说\"城市非随机\"? \n"
            "**反例**: 若用户自选城市呢? **假设**处理效应在城市间异质, ATE 还能简单均值差吗? \n"
            "**为什么** Day1 的 NSW 观测视角需要后门调整, Day2 的 RCT 视角不需要? \n"
            "**依据**是什么? 一句话总结。"
        )

    return feedback, next_scaffold

# 仿真 4 轮 (学生答案静态注入, 实际中学生输入)
student_answers = [
    "随机化使两组在所有特征上分布相同",           # round 1
    "MDE=1% 时 n 约 8200, MDE 减半 n 翻 4 倍",     # round 2
    "1-ρ² = 0.64, t 约变 1.25 倍",                 # round 3
    "DiD 适合非随机, RCT 适合随机",                # round 4
]

scaffold = 0
round_results = []
for r in range(1, MAX_ROUNDS + 1):
    ans = student_answers[r-1]
    fb, scaffold = socratic_turn(ans, r, scaffold)
    # 记录每轮是否需要降 scaffold (0=无需, 1=提示, 2=worked引用)
    round_results.append(scaffold)
    print(f"\n=== Round {r} (scaffold L{scaffold}) ===")
    print(f"[Student]: {ans}")
    print(f"[Fellow]: {fb}")

print("\n=== Tutorial 结束 - 进入 Cell 4 student_model + Cell 5 Hattie 反馈 ===")
print(f"round_results (0=独立答对, >0=需scaffold): {round_results}")


In [ ]:
# student_model.json 读写 (跨单元复用, 记录掌握度/盲点)
# Oxford tutorial 的 fellow 会在下次见面前读这个文件, 针对性回扣盲点
import json, os

MODEL_PATH = "student_model.json"

def load_student_model():
    if os.path.exists(MODEL_PATH):
        return json.load(open(MODEL_PATH, encoding='utf-8'))
    return {
        "student_id": "default",
        "units": {},
        "global_blindspots": [],
        "last_tutorial": None,
        "daily_usage": {}  # 限频: {date: count}
    }

def save_student_model(model):
    with open(MODEL_PATH, 'w', encoding='utf-8') as f:
        json.dump(model, f, ensure_ascii=False, indent=2)

def check_daily_limit(model, date_str, limit=1):
    """限频: 每生每天 1 次 tutorial (防 LLM 依赖)。"""
    used = model.get('daily_usage', {}).get(date_str, 0)
    return used < limit, limit - used

def update_after_tutorial(model, unit_id, round_results, blindspots, date_str):
    """tutorial 结束后更新 student_model。"""
    # 限频计数
    model.setdefault('daily_usage', {})
    model['daily_usage'][date_str] = model['daily_usage'].get(date_str, 0) + 1
    # 单元掌握度
    model.setdefault('units', {})
    mastery = 1.0 - (sum(round_results) / (len(round_results) * 2))  # 0=全需scaffold2, 1=全独立
    model['units'][unit_id] = {
        'mastery_estimate': round(mastery, 2),
        'blindspots': blindspots,
        'rounds_completed': len(round_results),
        'scaffold_levels_reached': round_results,
        'tutorial_date': date_str
    }
    # 全局盲点合并 (跨单元复用: Day3/Day5 的 fellow 会读这个列表)
    for b in blindspots:
        if b not in model['global_blindspots']:
            model['global_blindspots'].append(b)
    model['last_tutorial'] = unit_id
    save_student_model(model)
    return model

# 仿真: 本单元 tutorial 结束后写入
model = load_student_model()
ok, remaining = check_daily_limit(model, '2026-07-25', limit=1)
print(f"限频检查: 今日可用={ok}, 剩余次数={remaining}")
if ok:
    model = update_after_tutorial(
        model,
        unit_id='skill3-day2-ab-testing',
        round_results=[0, 0, 1, 0],  # 4 轮 scaffold 情况 (round 3 需提示)
        blindspots=['CUPED β 公式推导 (Cov/Var 不是 Cov/SD)', 'p<0.05 vs 业务意义区分', 'DiD 平行趋势假设未提及'],
        date_str='2026-07-25'
    )
    print(json.dumps(model, ensure_ascii=False, indent=2))
else:
    print("今日已达限频上限 (1次/天), 请 24h 后再来。建议先做 practice.md 的 drill D1-D3。")


## Hattie 四级 Formative Feedback (避免 Self 级表扬)

> Hattie (2007) RER 77(1):81-112 - 反馈的 3 个问题 (Feed Up / Feed Back / Feed Forward) × 4 个层级。
> 本 tutorial 按四级标注反馈, 刻意避免 Self 级表扬 (Hattie 研究表明 Self 级反馈与成绩弱相关甚至负相关)。

### [TASK] 任务级反馈 - 关于具体任务的反馈 (最强效)
- 你的 CUPED β 公式漏了中心化 (X-X̄), β 应该是 Cov(Y,X)/Var(X), 而不是 Cov(Y,X)/SD(X)。
- 均衡性检验中 age 的 t 统计量公式正确, 但 p 值解读错 -- p>0.05 不是"接受 H0", 而是"无法拒绝 H0"。
- 样本量计算数值正确 (8200), 但 MDE 减半翻 4 倍的推导跳步了, 应写出 n ∝ 1/MDE² 的反比平方关系。

### [PROCESS] 过程级反馈 - 关于理解任务的方式
- 你在 Round 2 跳过了 MDE 与 n 的反比平方关系, 直接给数字。**如何**先推导再算? 建议下次先写 n ∝ 1/MDE² 再代数。
- Round 4 中你用"DiD 适合非随机"作答, 这是结论而非推理。**为什么** DiD 需要平行趋势? 这个过程级假设你没提到。
- Round 3 你能写出 1-ρ² 但无法解释"为什么是 ρ² 而不是 ρ", 说明你记住了结论但没推导过程。建议重做 D3 阶段1。

### [SELF-REG] 自我调节级反馈 - 关于学生自我监控
- 你在 Round 3 触发了 scaffold L1 (给提示), 说明你意识到自己卡住并求助。这是好的 self-regulation。
- 但 Round 1 你没意识到"随机化"需要数学表达, 直接用模糊词。**如何**在下一单元提前自检"我的回答是结论还是论证"?
- 建议下次 tutorial 前用 diagnostic 自测, 若某题不确定就先做对应 drill 的阶段1, 不要带着盲点进 tutorial。

### [FEED-FORWARD] 前馈级反馈 - 关于下一步 (跨单元)
- 推荐复习: Day 1 (观测因果后门调整) - 对比 RCT vs 观测, 巩固 ILO4。
- 推荐间隔重复: schedule.json C1 (RCT 均值差=ATE) + C3 (CUPED) - 1 天后复习, FSRS-6 算法。
- 下次 tutorial (Day 3 准实验) 前, 请先完成 PSM/DiD 的 pre-task, 我们会追问"什么时候 RCT 不可行"。
- progressive_project M2 (milestone) 本周截止, 重点检查 TODO4 比例 Z 检验与 TODO5 事后功效。

> **刻意避免的反馈**: "做得好!"、"你很聪明"等 Self 级表扬 (Hattie effect size 低)。本反馈全部聚焦 Task/Process/Self-Reg/Feed-Forward。


## 限频与 Exit Artifact

### 限频 (防 LLM 依赖, Oxford tutorial 每周 1 次的节奏)
- **每生每天 1 次**: 本 tutorial 每天最多 1 次完整 Socratic loop (1次/天)。
- **为什么限频**: Oxford tutorial 每周 1 次, 不是每天。学生需要时间消化、retrieval practice (Butler 2010)、spaced repetition (FSRS-6)。每天频繁用 LLM 会形成依赖, 退化建构性。
- **超频处理**: 提示学生先做 practice.md 的 drill D1-D3, 24h 后再来。
- **每日上限**: 3 次重试同一 drill (与 practice.md retry_policy 一致)。
- **实现**: student_model.json 的 daily_usage 字段记录每日使用次数, check_daily_limit() 函数强制。

### Exit Artifact (tutorial 结束必须提交)

学生须在 student_model.json 中记录:

1. **2-3 个盲点** (本单元新发现的, 写入 units[unit_id].blindspots + global_blindspots):
   - 例: "CUPED β 公式推导 (Cov/Var 不是 Cov/SD)"
   - 例: "p<0.05 vs 业务意义区分"
   - 例: "DiD 平行趋势假设未提及"

2. **推荐复习单元** (跨单元 feed-forward, 写入 global_blindspots 供下个单元 fellow 读取):
   - 例: Day 1 (观测因果后门调整) - 巩固 ILO4
   - 例: Day 3 (准实验 PSM/DiD) - 预习方法选择
   - 例: Day 5 (规模实验) - CUPED 工业应用

3. **下次 pre-task 承诺**:
   - 例: "Day 3 前 24h 完成 PSM pre-task"
   - 例: "本周完成 progressive_project M2 milestone"

### 跨单元复用 (student_model.json 的核心价值)

student_model.json 会带入 Day 3 / Day 5 的 tutorial, fellow 会先读 blindspots 列表, 在新单元的 Socratic 追问中针对性回扣:

> Day 3 开场追问: "你 Day 2 说 CUPED 要求 X 不受 T 影响, 那 PSM 的匹配变量呢? 也是这个要求吗? **为什么**?"

这使整个学习材料包的 58 个单元形成连续的建构性学习路径, 而非孤立的知识点。
